In [1]:
# Proofs of Knowledge

from pwn import * # pip install pwntools
from Crypto.Util.number import long_to_bytes
import json

r = remote("socket.cryptohack.org", 13425)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

p = 0x1ed344181da88cae8dc37a08feae447ba3da7f788d271953299e5f093df7aaca987c9f653ed7e43bad576cc5d22290f61f32680736be4144642f8bea6f5bf55ef
q = 0xf69a20c0ed4465746e1bd047f57223dd1ed3fbc46938ca994cf2f849efbd5654c3e4fb29f6bf21dd6abb662e911487b0f9934039b5f20a23217c5f537adfaaf7
g = 2

assert p == g * q + 1

# we are prover now
w = 0x5a0f15a6a725003c3f65238d5f8ae4641f6bf07ebf349705b7f1feda2c2b051475e33f6747f4c8dc13cd63b9dd9f0d0dd87e27307ef262ba68d21a238be00e83
y = 0x514c8f56336411e75d5fa8c5d30efccb825ada9f5bf3f6eb64b5045bacf6b8969690077c84bea95aab74c24131f900f83adf2bfe59b80c5a0d77e8a9601454e5
assert y == pow(g, w, p)

rand = randint(0, q - 1)
a = pow(g, rand, p)
    
response = json_send({"a": a})
print(response)
e = response['e']

z = (rand + e * w) % q

response = json_send({"z": z})
print(response)

[x] Opening connection to socket.cryptohack.org on port 13425
[x] Opening connection to socket.cryptohack.org on port 13425: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13425: Done
b'Prove to me that you know an w such that g^w = y mod p. Send me a = g^r mod p for some random r in range(q)\n'
{'e': 2265892731403162543147362640073174109731484060584041742888993172370725480086763799901228701972880747108693969229913940850365503101866496356207401365356309, 'message': 'send me z = r + e*w mod q'}
{'flag': 'crypto{sigma_protocol_complete!}', 'message': 'You convinced me you know an `w` such that g^w = y mod p!'}


In [2]:
# Special Soundness

from pwn import * # pip install pwntools
from Crypto.Util.number import long_to_bytes
import json

r = remote("socket.cryptohack.org", 13426)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

p = 0x1ed344181da88cae8dc37a08feae447ba3da7f788d271953299e5f093df7aaca987c9f653ed7e43bad576cc5d22290f61f32680736be4144642f8bea6f5bf55ef
q = 0xf69a20c0ed4465746e1bd047f57223dd1ed3fbc46938ca994cf2f849efbd5654c3e4fb29f6bf21dd6abb662e911487b0f9934039b5f20a23217c5f537adfaaf7
g = 2

assert p == g * q + 1

# we are verifier now

response = json_send()
a1 = response['a']
y = response['y']
print(response['message'])

e1 = 2 #randint(0, 2**511-1)
response = json_send({'e': e1})
z1 = response['z']
print(response['message'])
assert pow(g, z1, p) == (a1 * pow(y, e1, p)) % p

response = json_send()
a2 = response['a2']
assert y == response['y']
print(response['message'])

e2 = 1 #randint(0, 2**511-1)
response = json_send({'e': e2})
z2 = response['z2']
print(response['message'])
assert pow(g, z2, p) == (a2 * pow(y, e2, p)) % p

# we know that z1 = (r + e1*flag) % q  and  z2 = (r + e2*flag) % q
# so (z1 - z2) % q = (e1 - e2)*flag % q
# if we will send e1=2 and e2=1 then flag = (z1 - z2) % q
flag = (z1 - z2) % q
print(long_to_bytes(flag))

[x] Opening connection to socket.cryptohack.org on port 13426
[x] Opening connection to socket.cryptohack.org on port 13426: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13426: Done
b'I will prove to you that I know flag `w` such that y = g^w mod p.\n'
send random e in range 0 <= e < 2^511
not convinced? I'll happily do it again!
send random e in range 0 <= e < 2^511
I hope you're convinced I know the flag now. Goodbye :)
b'crypto{specially_sound_sigmas}0\xa31\xa0U\xd7\x95$\xd0^OD:\xb6\xdb\xc6\xe6\xaf\xa7\x06W\x9d\xae\xd0+\xa4\x84\x14\xacd\xd8K'


In [ ]:
# Honest Verifier Zero Knowledge

from pwn import * # pip install pwntools
from Crypto.Util.number import long_to_bytes
import json

r = remote("socket.cryptohack.org", 13427)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

p = 0x1ed344181da88cae8dc37a08feae447ba3da7f788d271953299e5f093df7aaca987c9f653ed7e43bad576cc5d22290f61f32680736be4144642f8bea6f5bf55ef
q = 0xf69a20c0ed4465746e1bd047f57223dd1ed3fbc46938ca994cf2f849efbd5654c3e4fb29f6bf21dd6abb662e911487b0f9934039b5f20a23217c5f537adfaaf7
g = 2

assert p == g * q + 1

# we are prover now

response = json_send()
e = response['e']
y = response['y']
print(response['message'])

# now we know e and y, so we can choose z,a so that g^z = a*y^e mod p
# lets send z=0 and a = y^(-e) mod p 
a = pow(y, -e, p)
z = 0
response = json_send({'a': a, 'z': z})
print(response)

[x] Opening connection to socket.cryptohack.org on port 13427
[x] Opening connection to socket.cryptohack.org on port 13427: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13427: Done
b'Send me a transcript for my given `e` proving that you know the flag `w` such that y = g^w mod p\n'
send me your transcript
{'flag': 'crypto{so_honest_very_zero_knowledge}', 'message': 'You convinced me you know an `w` such that g^w = y mod p!'}


In [ ]:
# Non-Interactive

from pwn import * # pip install pwntools
from Crypto.Util.number import long_to_bytes,bytes_to_long
from hashlib import sha512
import json

r = remote("socket.cryptohack.org", 13428)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

p = 0x1ed344181da88cae8dc37a08feae447ba3da7f788d271953299e5f093df7aaca987c9f653ed7e43bad576cc5d22290f61f32680736be4144642f8bea6f5bf55ef
q = 0xf69a20c0ed4465746e1bd047f57223dd1ed3fbc46938ca994cf2f849efbd5654c3e4fb29f6bf21dd6abb662e911487b0f9934039b5f20a23217c5f537adfaaf7
g = 2
assert p == g * q + 1

# we are prover now
w = 0xdb968f9220c879b58b71c0b70d54ef73d31b1627868921dfc25f68b0b9495628b5a0ea35a80d6fd4f2f0e452116e125dc5e44508b1aaec89891dddf9a677ddc0
y = 0x1a1b551084ac43cc3ae2de2f89c6598a081f220010180e07eb62d0dee9c7502c1401d903018d9d7b06bff2d395c46795aa7cd8765df5ebe7414b072c8289170f0
assert y == pow(g, w, p)

response = json_send()
assert y == response['y']

rand = randint(0, q - 1)
a = pow(g, rand, p)
fiat_shamir_input = str(a).encode()
e = bytes_to_long(sha512(fiat_shamir_input).digest()) % 2**511
z = (rand + e * w) % q

response = json_send({'a': a, 'z': z})
print(response)


[x] Opening connection to socket.cryptohack.org on port 13428
[x] Opening connection to socket.cryptohack.org on port 13428: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13428: Done
b'Send me a nizk showing that you know `w` such that y = g^w mod p\n'
{'flag': 'crypto{shvzk_and_ss_to_nizk}', 'message': 'You convinced me you know an `w` such that g^w = y mod p!'}


In [ ]:
# Too Honest

from pwn import * # pip install pwntools
from Crypto.Util.number import long_to_bytes
import json

r = remote("socket.cryptohack.org", 13429)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

# N is composition of two primes
N = 63506177426384102189597350894327047299059434133653566917776601666605133716653510828029111986956978773016660313963972378811186153674164948861199369871734498221215139927864142313488277305751745855210473314367642273303159704466900274761354992859789827863358153922459760984397971477173435625199596782211170294424560686178858124003120741008270927463303483018910205943877584647744143454984243979284973117132536957364157878132874844783228762221620863204335896952103079109039534346621267709606103312376393511653638269034043434410564414042523141936372609708140474052147124354400977541403247799192906955295291389109531010594317
g = 2


# we are verifier now

response = json_send()
a = response['a']
y = response['y']
print(response['message'])

# we can not send two e
# we can start new session, but can not help us
# need to solve y = g^(-f) mod N, a = g^r mod N, z = (r + e*f) 

# after seeing hint I saw that since z = (rand + e*flag) then we can send big e so that it will shift flag and will not overlap with rand, so y is not needed at all
# maybe it is impossible to if server will calculate z = (r + e*flag) % N

k1 = 512
k2 = 128
S = 2**k1
e = 2**(2*k2+k1) # from server side code

response = json_send({'e': e})
z = response['z']
print(response['message'])
flag = z // e # shifting back

print(long_to_bytes(flag))

[x] Opening connection to socket.cryptohack.org on port 13429
[x] Opening connection to socket.cryptohack.org on port 13429: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13429: Done
b'I will prove to you that I know flag `w` such that y = g^-w mod N\n'
Send a random e in range 0 <= e < 2^{k2}
I hope you're convinced I know the flag now. Goodbye :)
b'crypto{2_hon3st_to_b3_tru3}\x15OL\xd2\xd2]\x13oV\xf3\x06\xe5\x8bR\x9f\x8f\xf9\xeej"\xbfF\xd1\xbe\xad.d\x1c\'\x7fyD..l'


In [33]:
# Mister Saplin's Preview

from pwn import * # pip install pwntools
from Crypto.Util.number import long_to_bytes, isPrime, inverse, GCD
import json

r = remote("socket.cryptohack.org", 13414)
print(r.readline())

def json_send(hsh={}):
    if len(hsh) != 0:
        request = json.dumps(hsh).encode()
        r.sendline(request)
    line = r.readline()
    return json.loads(line.decode())

json_send({"option":"get_nodes","nodes":"1,2;3,4"}) 

[x] Opening connection to socket.cryptohack.org on port 13414
[x] Opening connection to socket.cryptohack.org on port 13414: Trying 134.122.111.232
[+] Opening connection to socket.cryptohack.org on port 13414: Done
b'Welcome to the saplins previews system implementation!\n'


{'error': "You don't have enough credits!"}

In [50]:
%pip install py_ecc

In [ ]:
# Pairing-Based Cryptography

from py_ecc.optimized_bn128 import G1, G2, multiply, pairing, FQ, FQ2,FQ12
from py_ecc.typing import (
    Optimized_Point3D,
    optimized_bn128_FQ
)
from Crypto.Util.number import long_to_bytes
import ast

flag_binary = ""

with open("PairingBasedCryptography/output.txt", "r") as f:
    lines = f.readlines()
    for line in lines:

        # this part is llm generated which converts tuples to xG,yG,zG 
        [xG_coords, g2_coords, zG_coeffs] = ast.literal_eval(line)
        xG = (FQ(xG_coords[0]), FQ(xG_coords[1]), FQ(xG_coords[2]))
        
        x_coords, y_coords, z_coords = g2_coords
        X = FQ2([FQ(x_coords[0]), FQ(x_coords[1])])
        Y = FQ2([FQ(y_coords[0]), FQ(y_coords[1])])
        Z = FQ2([FQ(z_coords[0]), FQ(z_coords[1])])
        yG = (X, Y, Z)

        # Convert zG to FQ12 (assuming zG_coeffs is a list/tuple of 12 ints)
        zG = FQ12([FQ(c) for c in zG_coeffs])


        
        unbiased_zG = pairing(yG, xG)  
        if unbiased_zG == zG:
            flag_binary += "1"
        else:
            flag_binary += "0"

flag_int = int(flag_binary,2)
print(long_to_bytes(flag_int))

b'crypto{Pa1rings_R_Str0ng}'
